In [14]:
import requests
from bs4 import BeautifulSoup
import mysql.connector
import time
from datetime import datetime, timedelta

session = ''
db_config = {
    'user': 'root',
    'password': '',
    'host': 'localhost'
}

def check_db_creation():
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()
    cursor.execute("CREATE DATABASE IF NOT EXISTS fraud_check_db")
    cursor.execute("USE fraud_check_db")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS steadfast_fraud_check (
            checking_status VARCHAR(255),
            number VARCHAR(255),
            success INT,
            cancellation INT,
            fraud_id VARCHAR(255),
            name VARCHAR(255),
            details TEXT,
            user_ID INT,
            status INT
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS steadfast_fraud_alerts (
            number VARCHAR(255),
            name VARCHAR(255),
            details TEXT,
            last_update TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
        )
    """)

    cursor.close()
    conn.close()

def steadfast_login():
    global session
    login_page_url = "https://steadfast.com.bd/login"  # Login page URL
    login_url = "https://steadfast.com.bd/login"       # Login submission URL
    session = requests.Session()
    login_page_response = session.get(login_page_url)
    if login_page_response.status_code == 200:
        soup = BeautifulSoup(login_page_response.text, 'html.parser')
        csrf_token = soup.find('input', {'name': '_token'})['value']
        print(f"CSRF Token: {csrf_token}")
    else:
        print(f"Failed to fetch login page. Status code: {login_page_response.status_code}")
        exit()

    payload = {
        "_token": csrf_token,
        "email": "towfikahmed1029@gmail.com",  # Replace with your email
        "password": "gmail.com1029"            # Replace with your password
    }

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.45 Safari/537.36",
        "Referer": login_page_url,
        "Origin": "https://steadfast.com.bd"
    }
    login_response = session.post(login_url, data=payload, headers=headers)

    if login_response.status_code == 200:
        print("Login request sent successfully.")
        if "dashboard" in login_response.text.lower() or "logout" in login_response.text.lower():
            print("Login successful!")
        else:
            print("Login failed! Check the response content:")
            print(login_response.text)
    else:
        print(f"Login failed! Status code: {login_response.status_code}")
        print(login_response.text)

def get_numbers_not_done(cursor):
    query = """
        SELECT number
        FROM steadfast_fraud_check
        WHERE (checking_status != 'done' OR checking_status IS NULL OR checking_status = '')
        AND (number IS NOT NULL AND number != '')
    """
    cursor.execute(query)
    result = [row[0] for row in cursor.fetchall()]  # Extract numbers into a list
    return result

def steadfast_fraud_check():
    print("Checking Number....")
    global session

    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()
    numbers = get_numbers_not_done(cursor)
    
    for Fraud_checking_number in numbers:
        print(f"Checking number: {Fraud_checking_number}")
        cookies = session.cookies.get_dict()  # Get all cookies
        xsrf_token = cookies.get('XSRF-TOKEN')  # Extract XSRF-TOKEN
        print(xsrf_token)
        session_token = cookies.get('steadfast_merchant_session')  # Extract steadfast_merchant_session
        cf_clearance = cookies.get('cf_clearance')
        cookies = {
            'XSRF-TOKEN': xsrf_token ,
            'steadfast_merchant_session': session_token,
            'cf_clearance': cf_clearance,
        }

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.45 Safari/537.36",
            'Referer': 'https://steadfast.com.bd/user/frauds/check',
            "Origin": "https://steadfast.com.bd"
        }
        response = session.get(f'https://steadfast.com.bd/user/frauds/check/{Fraud_checking_number}', cookies=cookies, headers=headers)

        if response.status_code == 200:
            print("Data fetched successfully!")  

            json_data = response.json()
            print(json_data)

            try:
                success = json_data[0]
                cancellation = json_data[1]
                fraud_id = json_data[2][0]['id']
                name = json_data[2][0]['name']
                details = json_data[2][0]['details']
                user_id = json_data[2][0]['user_id']
                status = json_data[2][0]['status']
            except Exception as e:
                success = 0
                cancellation = 0
                fraud_id = 0
                name = ''
                details = ''
                user_id = 0
                status = 0
                print(e)
            cursor.execute("""
                UPDATE steadfast_fraud_check
                SET checking_status = 'done',success = %s,cancellation = %s,fraud_id = %s,name = %s,details = %s,user_ID = %s,status = %s
                WHERE number = %s
            """, (
                success,cancellation,fraud_id,name,details,user_id,status,Fraud_checking_number
            ))
            conn.commit()
        
        else:
            steadfast_login()

    cursor.close()
    conn.close()
    print("Number Checked..")

def collect_page_data(page_number):
    global session
    cookies = session.cookies.get_dict()  # Get all cookies
    xsrf_token = cookies.get('XSRF-TOKEN')  # Extract XSRF-TOKEN
    session_token = cookies.get('steadfast_merchant_session')  # Extract steadfast_merchant_session
    cf_clearance = cookies.get('cf_clearance')

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.45 Safari/537.36",
        'Referer': 'https://steadfast.com.bd/user/frauds/recent',
        "Origin": "https://steadfast.com.bd"
    }
    if page_number == 1:
        page_number = ''
    params = {'page': page_number}

    cookies = {
        'XSRF-TOKEN': xsrf_token ,
        'steadfast_merchant_session': session_token,
        'cf_clearance': cf_clearance,
    }

    response = session.get("https://steadfast.com.bd/user/frauds/recent", params=params, headers=headers,cookies=cookies)
    page_data = []
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        all_comments = soup.find_all('div', class_ = 'fraud-list d-flex flex-column gap-3')
        print(f"Total {len(all_comments)} Fraud found in page {page_number} . ")
        for com in all_comments:
            try:
                phone = com.find_all("div" , class_ ="cell_2")[0].find("p").text.strip()
                name = com.find_all("div" , class_ ="cell_2")[1].find("p").text.strip()
                details = com.find_all("div" , class_ ="cell_2")[2].find("p").text.strip()
                entry_time = com.find("span" , class_ ="text-end").text.strip()
                extracted_data = [phone,name,details,entry_time]
                page_data.append(extracted_data)
            except:pass
        return page_data

def get_last_number_and_time(cursor):
    query = """
        SELECT number, last_update
        FROM steadfast_fraud_alerts
        ORDER BY last_update DESC
        LIMIT 1
    """
    cursor.execute(query)
    result = cursor.fetchone()
    if result:
        # Format last_update to string for consistency with insert format
        last_update_str = result[1].strftime('%d-%m-%Y %I:%M %p')  # Access `last_update` using index 1
        return result[0], last_update_str  # Access `number` using index 0
    else:
        # If no data, return the default values (last date = 3 months ago, number = '999')
        three_months_ago = datetime.now() - timedelta(days=90)
        three_months_ago_str = three_months_ago.strftime('%d-%m-%Y %I:%M %p')
        return '999', three_months_ago_str

def update_steadfast_fraud_alerts(conn,cursor, all_details):
    phone = all_details[0]
    name = all_details[1]
    details = all_details[2]
    last_update = all_details[3]  # its string format example data = '12-01-2025 03:34 pm'
    last_update_dt = datetime.strptime(last_update, '%d-%m-%Y %I:%M %p')
    insert_query = """
        INSERT INTO steadfast_fraud_alerts (number, name, details, last_update)
        VALUES (%s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            name = VALUES(name),
            details = VALUES(details),
            last_update = VALUES(last_update)
    """
    cursor.execute(insert_query, (phone, name, details, last_update_dt))
    conn.commit()  
    return True

def steadfast_fraud_alerts():
    print("Collecting activity details by pagination...")
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()
    last_number,last_time = get_last_number_and_time(cursor)
    page_number = 0
    collection_complete = False
    if last_number != '999':
        last_time_dt = datetime.strptime(last_time, '%d-%m-%Y %I:%M %p')
    else:
        last_time_dt = datetime.now() - timedelta(days=90)
    while True:
        try:
            page_number += 1
            print(f"Page number : {page_number}")
            nested_details = collect_page_data(page_number)
            for all_details in nested_details:
                current_time_dt = datetime.strptime(all_details[-1], '%d-%m-%Y %I:%M %p')
                if all_details[0] == last_number and current_time_dt == last_time_dt:
                    collection_complete = True
                    break
                if last_number == '999' and current_time_dt < last_time_dt:
                    collection_complete = True
                    break
                update_steadfast_fraud_alerts(conn,cursor, all_details)
            if collection_complete:
                print("No more data found in website.")
                break

        except:
            steadfast_login()
            break
        
    cursor.close()
    conn.close()
    
check_db_creation()
db_config = {
    'user': 'root',
    'password': '',
    'host': 'localhost',
    'database': 'fraud_check_db'  
}

steadfast_login()



CSRF Token: iAGWWxIsX9Vf4fikOojmti9XKqiID9ADLkJ9lxG8
Login request sent successfully.
Login successful!


In [ ]:
pip install mysql-connector

In [15]:
steadfast_fraud_check()

# steadfast_fraud_alerts()

Checking Number....
Checking number: 01706972285
eyJpdiI6Ik5WaDc0WGpoY3ZwRGloZjlNN0ZQOXc9PSIsInZhbHVlIjoiRTE2c2RUSUtBUzRScVRnV1ppRERGS2xxWUY0LzlCRWRSa0tPc3dSRVZpV0JYS3RqL054azk1ckFwSTFTVHU5QjBLZ1dWZFdYNnVvdHRUd1VteXZ2NlhYYzZLeVVoaVlBVFZzWVNBTXFLV1E3UTJMdnFUQ0QxTTFCRzZXOUlXTnQiLCJtYWMiOiJmYjM2N2RiMzgwMmQwZDU2ZjY3NmRlOWE1ZjRjZDgyZmZjYmY3MjBhMWFjODJjZDVhY2I5YjgwMTlhMjRkYWY5IiwidGFnIjoiIn0%3D
Data fetched successfully!
[4, 0, []]
list index out of range
Number Checked..
